In [1]:
from google.cloud import bigquery
from google.colab import auth
import pandas as pd

In [2]:
auth.authenticate_user()
client = bigquery.Client(project='extended-altar-423112-j9')
query = """
SELECT
    transac.transaction_id,
    transac.unit_price,
    dt.week_day,
    dt.is_holiday,
    pro.promotion_type,
    wth.weather_conditions,
    transac.actual_demand,
FROM `extended-altar-423112-j9.Walmart.fact_transaction` transac
JOIN `extended-altar-423112-j9.Walmart.dim_date` dt ON transac.date_id = dt.date_id
LEFT JOIN `extended-altar-423112-j9.Walmart.dim_promotion` pro ON transac.promotion_id = pro.promotion_id
LEFT JOIN `extended-altar-423112-j9.Walmart.dim_weather` wth ON transac.weather_id = wth.weather_id
"""
query_result = client.query(query)
print("Extract machine learning features from BigQuery complete!")

Extract machine learning features from BigQuery complete!


In [3]:
df_ml = query_result.to_dataframe()
print(f"Dataset ready! The dataset has {df_ml.shape[0]} rows and {df_ml.shape[1]} columns")
df_ml.head()

Dataset ready! The dataset has 5000 rows and 7 columns


,transaction_id,unit_price,week_day,is_holiday,promotion_type,weather_conditions,actual_demand
0,387,1484.97,Wednesday,True,None,Rainy,418
1,2065,1179.33,Saturday,False,None,Sunny,146
2,2864,1142.90,Saturday,True,None,Rainy,210
3,1906,1723.04,Thursday,False,None,Rainy,472
4,2894,1346.81,Monday,True,None,Cloudy,422


In [4]:
print("Encoding machine learning features...")
df_ml['is_holiday'] = df_ml['is_holiday'].astype(int)
df_encoded = pd.get_dummies(df_ml, columns = ['weather_conditions','promotion_type', 'week_day'], drop_first=True) 
df_encoded.drop(columns='transaction_id', inplace = True)
print(f"Encoding completed. Encoded dataframe has {df_encoded.shape[0]} rows and {df_encoded.shape[1]} columns")
df_encoded

Encoding machine learning features...
Encoding completed. Encoded dataframe has 5000 rows and 14 columns


,unit_price,is_holiday,actual_demand,weather_conditions_Rainy,weather_conditions_Stormy,weather_conditions_Sunny,promotion_type_None,promotion_type_Percentage Discount,week_day_Monday,week_day_Saturday,week_day_Sunday,week_day_Thursday,week_day_Tuesday,week_day_Wednesday
0,1484.97,1,418,True,False,False,True,False,False,False,False,False,False,True
1,1179.33,0,146,False,False,True,True,False,False,True,False,False,False,False
2,1142.90,1,210,True,False,False,True,False,False,True,False,False,False,False
3,1723.04,0,472,True,False,False,True,False,False,False,False,True,False,False
4,1346.81,1,422,False,False,False,True,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,543.65,0,313,False,False,False,False,True,True,False,False,False,False,False
4996,825.84,1,287,False,False,False,True,False,False,True,False,False,False,False
4997,244.55,0,286,False,False,False,True,False,False,True,False,False,False,False
4998,552.82,0,272,True,False,False,True,False,False,False,False,False,True,False


In [5]:
from sklearn.model_selection import train_test_split 

X = df_encoded.drop(columns = 'actual_demand')
y = df_encoded['actual_demand']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train set rows (80%): {X_train.shape[0]}")
print(f"Test set rows (20%): {X_test.shape[0]}")

Train set rows (80%): 4000
Test set rows (20%): 1000


In [6]:
from sklearn.ensemble import RandomForestRegressor

print("Running Random Forest...")
model = RandomForestRegressor(n_estimators = 100, random_state=67)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print("Prediction completed!")
print("Here is a peek of the first ten predictions")
for i in range(10):
    prediction_value = round(predictions[i])
    actual_value = y_test.iloc[i] 
    print(f"The model prediction is {prediction_value}, The actual value was {actual_value}")

Running Random Forest...
Prediction completed!
Here is a peek of the first ten predictions
The model prediction is 227, The actual value was 225
The model prediction is 263, The actual value was 424
The model prediction is 350, The actual value was 201
The model prediction is 240, The actual value was 415
The model prediction is 309, The actual value was 419
The model prediction is 194, The actual value was 186
The model prediction is 276, The actual value was 464
The model prediction is 257, The actual value was 262
The model prediction is 259, The actual value was 338
The model prediction is 310, The actual value was 241


In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
print(f"Mean absolute error: {mean_absolute_error(y_test, predictions):.4f}")
print(f"Root Squared mean error: {np.sqrt(mean_squared_error(y_test, predictions)):.4f}")
print(f"R squared: {r2_score(y_test, predictions):.4f}")

Mean absolute error: 111.8003
Root Squared mean error: 132.4430
R squared: -0.1571


In [ ]:
from google.colab import drive
import shutil
import joblib

print("Extracting model...")
joblib.dump(model,"demand_model.pkl")
print("Extracting completed!")

print("1. Connecting to Google Drive...")
drive.mount('/content/drive')
print("2. Copying the model to Drive...")
shutil.copy('demand_model.pkl', '/content/drive/MyDrive/demand_model.pkl')

print("Success! The file is now in Google Drive.")

Extracting model...
Building API server...
API server set up completed!
Simulating front end API communication...
Response status: 200
API JSON: 181.6
